In [ ]:
from datasets import load_dataset
import numpy as np
import torch
import os
import gradio as gr



# news_bert_gradio.py
# Fine-tune BERT on AG News and serve a Gradio demo.
# Usage: python news_bert_gradio.py

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import sklearn.metrics as metrics

def main():
    model_name = "bert-base-uncased"
    dataset = load_dataset("ag_news")
    labels = dataset["train"].features["label"].names

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def preprocess(batch):
        return tokenizer(batch["text"], truncation=True)
    tokenized = dataset.map(preprocess, batched=True, remove_columns=["text"])

    # create small validation split from train
    tokenized["train"] = tokenized["train"].train_test_split(test_size=0.1, seed=42)["train"]
    tokenized["validation"] = tokenized["train"].train_test_split(test_size=0.1111, seed=42)["test"]  # approx 10% overall
    # If above splitting logic is awkward, fallback to using dataset['test'] as validation:
    tokenized["validation"] = tokenized["test"]

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(labels))

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    def compute_metrics(eval_pred):
        logits, labels_arr = eval_pred
        preds = np.argmax(logits, axis=-1)
        acc = metrics.accuracy_score(labels_arr, preds)
        f1 = metrics.f1_score(labels_arr, preds, average="weighted")
        return {"accuracy": acc, "f1_weighted": f1}

    training_args = TrainingArguments(
        output_dir="./news-bert",
        evaluation_strategy="epoch",
        save_strategy="epoch",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        logging_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="accuracy",
        seed=42,
        fp16=torch.cuda.is_available(),
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_results = trainer.evaluate(tokenized["test"])
    print("Test eval:", eval_results)

    model_dir = "./news-bert-final"
    os.makedirs(model_dir, exist_ok=True)
    trainer.save_model(model_dir)
    tokenizer.save_pretrained(model_dir)

    # Gradio demo
    device = 0 if torch.cuda.is_available() else -1
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()
    if torch.cuda.is_available():
        model.to("cuda")

    def predict(text):
        inputs = tokenizer(text, truncation=True, return_tensors="pt")
        if torch.cuda.is_available():
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()[0]
        return {labels[i]: float(probs[i]) for i in range(len(labels))}

    iface = gr.Interface(
        fn=predict,
        inputs=gr.Textbox(lines=3, placeholder="Enter a news headline or short text..."),
        outputs=gr.Label(num_top_classes=len(labels)),
        title="AG News Topic Classifier (BERT)",
        description="Fine-tuned bert-base-uncased on AG News. Returns topic probabilities.",
    )
    iface.launch(server_name="0.0.0.0", share=False, inbrowser=True)

if __name__ == "__main__":
    main()